In [29]:
import time
import numpy as np
from pynq import Overlay, allocate

# 1. Load the bitstream
ol = Overlay("matmul_design.bit")
print("Overlay loaded successfully!")

# 2. Grab handles to the IP blocks
dma = ol.axi_dma_0
matmul_ip = ol.matmul_stream_0

# 3. Define Project Parameters
MATRIX_SIZE = 64
NUM_VECTORS = 10  # Processing 10,000 vectors in one shot

Overlay loaded successfully!


In [30]:
# 1. Generate the constant 64x64 Matrix and input batch as standard floats
# (We keep these to calculate our golden reference later)
A_matrix_float = np.random.uniform(1,10, size=(MATRIX_SIZE, MATRIX_SIZE)).astype(np.float32)
X_batch_float = np.random.uniform(1, 10, size=(NUM_VECTORS, MATRIX_SIZE)).astype(np.float32)
print(A_matrix_float)
print(X_batch_float)
# 2. CONVERT TO FIXED-POINT (ap_fixed<24, 14>)
# Multiply by 2^10 to shift the fractional bits into the integer space, then cast to int32
FRACTIONAL_BITS = 10
A_fixed = (A_matrix_float * (2**FRACTIONAL_BITS)).astype(np.int32)
X_fixed = (X_batch_float * (2**FRACTIONAL_BITS)).astype(np.int32)

# 3. Configure the Hardware Registers via AXI-Lite
# (Verify these offsets in your xmatmul_stream_hw.h!)
matmul_ip.write(0x10, MATRIX_SIZE)
matmul_ip.write(0x18, MATRIX_SIZE)
matmul_ip.write(0x20, NUM_VECTORS)

# Write the fixed-point A matrix into the IP's BRAM
matmul_ip.write(0x0100, A_fixed.flatten('F').tobytes()) 

print("Fixed-point Matrix A and parameters loaded into hardware.")

[[7.8946915 6.280068  4.8173203 ... 5.83586   1.6973592 1.8936491]
 [3.5864384 2.5741296 2.082351  ... 3.1192691 2.8840075 5.485646 ]
 [6.956378  3.5335157 2.566168  ... 4.3809247 9.532397  1.530405 ]
 ...
 [2.2456894 7.1045375 2.0308182 ... 1.2646923 8.441852  1.0561154]
 [8.141773  3.0197504 6.268656  ... 8.906895  9.135521  8.123076 ]
 [8.871263  2.3037887 8.85915   ... 7.154055  9.034794  1.4295622]]
[[2.630866  3.9816282 9.00092   5.3345733 7.3380713 2.087365  3.0247464
  6.0218496 5.090722  6.424954  7.4040947 2.2833736 4.7920985 4.6858068
  4.7621727 4.5939846 3.2146595 8.24597   5.875741  1.8813101 3.360256
  4.370026  7.218952  5.3729167 8.900069  7.3781176 5.510262  5.8849845
  3.7606783 8.448333  9.034896  6.949431  3.5681489 1.2998115 5.447097
  8.117883  7.1971326 7.9001565 5.0778193 6.8125243 1.7859815 7.0408697
  2.1071339 1.3188016 8.7012205 6.3515906 7.4780645 4.300731  3.1537905
  2.9685848 6.0814743 3.3151028 4.16558   6.581759  5.7432218 6.7905354
  6.6915374 5.9758

In [31]:
# 1. Allocate Contiguous Memory Buffers for the DMA
total_elements = NUM_VECTORS * MATRIX_SIZE

# CRITICAL: Buffers must be int32 because the DMA is moving raw fixed-point bits
in_buf = allocate(shape=(total_elements,), dtype=np.int32)
out_buf = allocate(shape=(total_elements,), dtype=np.int32)

# 2. Copy our 2D fixed-point batch into the 1D flat input buffer
np.copyto(in_buf, X_fixed.flatten())

# 3. Start the Hardware IP
matmul_ip.write(0x00, 0x01) # ap_start

print("Starting massive DMA transfer...")
t0 = time.perf_counter()

# 4. Trigger DMA transfers
dma.sendchannel.transfer(in_buf)
dma.recvchannel.transfer(out_buf)

# 5. Wait for the hardware to finish the entire batch
dma.sendchannel.wait()
dma.recvchannel.wait()

t1 = time.perf_counter()
hw_time = t1 - t0

print(f"Hardware finished processing {NUM_VECTORS} vectors in {hw_time*1000:.2f} ms!")

Starting massive DMA transfer...
Hardware finished processing 10 vectors in 3.01 ms!


In [32]:
# 1. Partition the flat integer output back into a 2D matrix
Y_hw_raw_int = np.array(out_buf).reshape(NUM_VECTORS, MATRIX_SIZE)

Y_hw_raw_int = np.left_shift(Y_hw_raw_int, 8)
Y_hw_raw_int = np.right_shift(Y_hw_raw_int, 8) 
'''to get sign bits correctly, because if we use pack function, to pack 24 bit ap_fixed 
to 32 bit ap_int and pad the left side with 0, when we read it back, it will interpret negative
24 bit number as positive 32 bit number instead(cus left side is zeroes not 1s)'''

# 2. CONVERT BACK TO FLOAT
# Cast to float32 and divide by 2^10 to restore the decimal point
Y_hw_matrix = Y_hw_raw_int.astype(np.float32) / (2**FRACTIONAL_BITS)

# 3. Calculate the Golden Reference in Software (using our original pure floats)
print("Calculating golden reference on ARM CPU...")
t_cpu_start = time.perf_counter()

# Y = X @ A.T (Batch matrix multiplication)
Y_ref_matrix = X_batch_float @ A_matrix_float.T 

t_cpu_end = time.perf_counter()
cpu_time = t_cpu_end - t_cpu_start

# Compare the entire matrices
max_diff = np.max(np.abs(Y_hw_matrix - Y_ref_matrix))

print("-" * 40)
# Tolerance relaxed to 0.05 because fixed-point math loses tiny fractions during MAC operations
if max_diff < 0.5: 
    print(f"SUCCESS! Hardware matches Software. (Max diff: {max_diff:.2e})")
else:
    print(f"FAIL! Outputs differ. (Max diff: {max_diff:.2e})")
print("-" * 40)

# Print Throughput Results
print(f"ARM CPU Time:  {cpu_time*1000:.2f} ms")
print(f"FPGA DMA Time: {hw_time*1000:.2f} ms")
print(f"Speedup:       {cpu_time / hw_time:.2f}x")
print(f"Throughput:    {NUM_VECTORS / hw_time:,.0f} vectors/second")

# Free memory to prevent memory leaks on the Pynq board
in_buf.freebuffer()
out_buf.freebuffer()

Calculating golden reference on ARM CPU...
----------------------------------------
SUCCESS! Hardware matches Software. (Max diff: 4.57e-01)
----------------------------------------
ARM CPU Time:  1.39 ms
FPGA DMA Time: 3.01 ms
Speedup:       0.46x
Throughput:    3,323 vectors/second
